In [ ]:
#%pip install -U ipython-sql sqlalchemy psycopg2-binary prettytable==3.11.0
#%pip install -U jupysql sqlalchemy psycopg2-binary prettytable
#!pip install openpyxl  
# import sys 
# !{sys.executable} -m pip install xlrd openpyxl --upgrade

In [1]:
%load_ext sql
%config SqlMagic.autopandas = False
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [8]:
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)
Magic= %sql $DATABASE_URL

In [31]:
%%sql
SET search_path TO "Superstore";

++
||
++
++

In [ ]:
excelfile = pd.ExcelFile(r'E:\DEPI Professional Data Analyst\Techincal\Mini_Project2\Superstore 2019.xls')


In [ ]:
S_Name=pd.ExcelFile(r'E:\DEPI Professional Data Analyst\Techincal\Mini_Project2\Superstore 2019.xls')
print(S_Name.sheet_names)

In [ ]:
Orders1 = pd.read_excel(excelfile, sheet_name="Orders")
Returns1 = pd.read_excel(excelfile, sheet_name="Returns").drop_duplicates()

In [ ]:
Orders1.columns
Returns1.columns

In [ ]:
Orders1.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in Orders1.columns]
Returns1.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in Returns1.columns]


In [ ]:
Orders1.info()
Returns1.info()

In [ ]:
Orders1.to_sql(
    name='orders',
    schema='Superstore',
    con=engine,
    if_exists='append',   # 'replace', 'append', or 'fail'
    index=False
)

engine.dispose()
print("Data saved to database")

In [ ]:
Returns1.to_sql(
    name='returns',
    schema='Superstore',
    con=engine,
    if_exists='append',   # 'replace', 'append', or 'fail'
    index=False
)

engine.dispose()
print("Data saved to database")

In [24]:
%%sql

-- 1) dim_customer
create TABLE dim_customer (
customer_Key SERIAL PRIMARY KEY,
customer_id TEXT UNIQUE NOT NULL,
customer_name TEXT NOT NULL,
segment TEXT NOT NULL
);

++
||
++
++

In [ ]:
%%sql

-- 2) dim_product
CREATE TABLE dim_product(
Product_Key SERIAL PRIMARY KEY, 
product_id TEXT UNIQUE NOT NULL,
product_name TEXT NOT NULL,
category TEXT NOT NULL,
sub_category TEXT NOT NULL
);

In [ ]:
%%sql

-- 3) dim_location
CREATE TABLE dim_location (
location_key SERIAL PRIMARY KEY,
city TEXT NOT NULL,
state TEXT NOT NULL,
postal_code int,
region TEXT NOT NULL,
country TEXT NOT NULL,
UNIQUE (city,state,postal_code,region)
);

In [ ]:
%%sql

-- 4) dim_ship_mode
create TABLE dim_ship_mode (
ship_mode_key SERIAL PRIMARY KEY,
ship_mode TEXT UNIQUE NOT NULL
);

In [ ]:
%%sql

-- 5) dim_date  
create TABLE dim_date (
date_key INT PRIMARY KEY,
full_date  DATE UNIQUE NOT NULL,
day        INT NOT NULL,
month      INT NOT NULL,
month_name TEXT NOT NULL,
quarter    INT NOT NULL,
year       INT NOT NULL,
weekday_name TEXT NOT NULL
);

In [ ]:
%%sql

-- 6) fact_sales
CREATE TABLE fact_sales (
    sales_key      SERIAL PRIMARY KEY,
    row_id         INT UNIQUE NOT NULL,      
    order_id       TEXT NOT NULL,
    customer_key   INT NOT NULL REFERENCES dim_customer(customer_key),
    product_key    INT NOT NULL REFERENCES dim_product(product_key),
    location_key   INT NOT NULL REFERENCES dim_location(location_key),
    ship_mode_key  INT NOT NULL REFERENCES dim_ship_mode(ship_mode_key),
    order_date_key INT NOT NULL REFERENCES dim_date(date_key),
    ship_date_key  INT NOT NULL REFERENCES dim_date(date_key),
    sales          NUMERIC NOT NULL,
    quantity       INT NOT NULL,
    discount       NUMERIC NOT NULL,
    profit         NUMERIC NOT NULL,
    is_returned    BOOLEAN NOT NULL DEFAULT FALSE
);

In [ ]:
%%sql

-- insert the data into the Dim-Tables:
insert into dim_customer (customer_id,customer_name,segment) 
SELECT DISTINCT customer_id , customer_name , segment 
FROM orders;

insert into dim_product (product_id , product_name , category , sub_category) 
SELECT DISTINCT on (product_id) product_id , product_name , category , sub_category 
FROM orders
ORDER BY product_id, product_name;

-- to clear the Table after the duplicate error 
TRUNCATE Table dim_product CASCADE;

-- to check the duplication issue
select product_id , count(DISTINCT product_name) as Names , count(DISTINCT category) as cat
FROM orders
GROUP BY product_id
HAVING count(DISTINCT product_name)>1 or count(DISTINCT category)>1
ORDER BY Names DESC
LIMIT 10;

insert into dim_location (city,state,postal_code,region,country) 
SELECT DISTINCT city, state, postal_code, region,'country_region'
FROM orders;

INSERT INTO dim_ship_mode (ship_mode)
SELECT DISTINCT ship_mode
FROM orders;

insert into dim_date (date_key,full_date,day,month,month_name,quarter,year,weekday_name)
SELECT 
    to_char (d,'YYYYMMDD')::int as date_key,
    d::DATE as full_date,
    extract(day from d)::int as day,
    extract(month from d)::int as month,
    to_char (d,'month') as month_name,
    extract(quarter from d)::int as quarter,
    extract(YEAR from d)::int as year,
    to_char (d,'day') as weekday_name
FROM generate_series(
    (SELECT MIN(order_date) from orders),
    (SELECT max(ship_date) from orders),
    INTERVAL '1 day') as d;


INSERT INTO fact_sales (
    row_id, order_id, customer_key, product_key, location_key,
    ship_mode_key, order_date_key, ship_date_key,
    sales, quantity, discount, profit, is_returned
)
SELECT
    o.row_id,
    o.order_id,
    c.customer_key,
    p.product_key,
    l.location_key,
    sm.ship_mode_key,
    TO_CHAR(o.order_date, 'YYYYMMDD')::INT,
    TO_CHAR(o.ship_date, 'YYYYMMDD')::INT,
    o.sales,
    o.quantity,
    o.discount,
    o.profit,
    CASE WHEN r.order_id IS NOT NULL THEN TRUE ELSE FALSE END
FROM orders o
JOIN dim_customer  c  ON c.customer_id = o.customer_id
JOIN dim_product   p  ON p.product_id  = o.product_id
JOIN dim_location  l  ON l.city = o.city AND l.state = o.state
                      AND (l.postal_code = o.postal_code OR (l.postal_code IS NULL AND o.postal_code IS NULL))
                      AND l.region = o.region
JOIN dim_ship_mode sm ON sm.ship_mode = o.ship_mode
LEFT JOIN returns  r  ON r.order_id = o.order_id;

SELECT COUNT(*) FROM dim_product;

In [53]:
%%sql

CREATE VIEW superstore.vw_profitability_report AS
SELECT
    d.year,
    l.region,
    p.category,
    p.sub_category,
    COUNT(f.sales_key)          AS total_orders,
    SUM(f.sales)                AS total_sales,
    SUM(f.profit)               AS total_profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0) * 100, 2) AS profit_margin_pct
FROM superstore.fact_sales f
JOIN superstore.dim_date    d ON d.date_key    = f.order_date_key
JOIN superstore.dim_location l ON l.location_key = f.location_key
JOIN superstore.dim_product  p ON p.product_key  = f.product_key
GROUP BY d.year, l.region, p.category, p.sub_category;


RuntimeError: (psycopg2.errors.DuplicateTable) relation "vw_profitability_report" already exists

[SQL: CREATE VIEW superstore.vw_profitability_report AS
SELECT
    d.year,
    l.region,
    p.category,
    p.sub_category,
    COUNT(f.sales_key)          AS total_orders,
    SUM(f.sales)                AS total_sales,
    SUM(f.profit)               AS total_profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0) * 100, 2) AS profit_margin_pct
FROM superstore.fact_sales f
JOIN superstore.dim_date    d ON d.date_key    = f.order_date_key
JOIN superstore.dim_location l ON l.location_key = f.location_key
JOIN superstore.dim_product  p ON p.product_key  = f.product_key
GROUP BY d.year, l.region, p.category, p.sub_category;]
(Background on this error at: https://sqlalche.me/e/21/f405)


In [54]:
%%sql

CREATE OR REPLACE FUNCTION superstore.get_customer_summary(p_customer_id text)
RETURNS TABLE (
    customer_name    text,
    segment          text,
    total_orders     BIGINT,
    total_sales      NUMERIC,
    total_profit     NUMERIC,
    returned_orders  BIGINT
)
LANGUAGE plpgsql
AS $$
BEGIN
    RETURN QUERY
    SELECT
        c.customer_name,
        c.segment,
        COUNT(f.sales_key)                              AS total_orders,
        SUM(f.sales)                                    AS total_sales,
        SUM(f.profit)                                   AS total_profit,
        SUM(CASE WHEN f.is_returned THEN 1 ELSE 0 END)  AS returned_orders
    FROM fact_sales f
    JOIN dim_customer c ON c.customer_key = f.customer_key
    WHERE c.customer_id = p_customer_id
    GROUP BY c.customer_name, c.segment;
END;
$$;

--SELECT customer_id FROM dim_customer LIMIT 10;
--SELECT * FROM get_customer_summary('RD-19930');

++
||
++
++

In [55]:
%%sql

--Q1--Total Sales and Profit by Category and Sub-category:
select d.category,d.sub_category, round(sum(sales),0) as Total_sales , round(sum(profit),2) as total_profit,
 ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0) * 100, 2)  AS profit_margin_pct
FROM superstore.fact_sales f
JOIN superstore.dim_product d on f.product_key = d.product_key
GROUP BY d.category, d.sub_category
ORDER BY total_profit DESC;
-- Q1.1-Total Sales and Profit by Category:
select DISTINCT d.category, round(sum(sales),0) as Total_sales , round(sum(profit),2) as total_profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0) * 100, 2)  AS profit_margin_pct
FROM superstore.fact_sales f
JOIN superstore.dim_product d on f.product_key = d.product_key
GROUP BY d.category
ORDER BY total_profit DESC;

category,total_sales,total_profit,profit_margin_pct
Technology,836154,145454.95,17.40
Office Supplies,719047,122490.80,17.04
Furniture,742000,18451.27,2.49


In [56]:
%%sql

--Q2-Top 10 Most Profitable Products:
select d.product_name,d.category, round(sum(sales),0) as Total_sales , round(sum(profit),2) as total_profit
FROM superstore.fact_sales f
JOIN superstore.dim_product d on f.product_key = d.product_key
GROUP BY d.product_name, d.category
ORDER BY total_profit DESC
LIMIT 10;


product_name,category,total_sales,total_profit
Canon imageCLASS 2200 Advanced Copier,Technology,61600,25199.93
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,Office Supplies,27453,7753.04
Hewlett Packard LaserJet 3310 Copier,Technology,18840,6983.88
Canon PC1060 Personal Laser Copier,Technology,11620,4570.93
Logitech G19 Programmable Gaming Keyboard,Technology,13757,4425.34
"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",Technology,18375,4094.98
Ativa V4110MDD Micro-Cut Shredder,Technology,7700,3772.95
"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14300,3717.97
Ibico EPK-21 Electric Binding System,Office Supplies,15876,3345.28
Zebra ZM400 Thermal Label Printer,Technology,6966,3343.54


In [57]:
%%sql

--Q3-Regions/states incurring losses (negative profit):
SELECT state ,total_profit from (
                select d.state, round(sum(profit),2) as total_profit
                FROM superstore.fact_sales f
                JOIN superstore.dim_location d on f.location_key = d.location_key
                GROUP BY d.state
)
WHERE total_profit < 0
ORDER BY total_profit ASC;

state,total_profit
Texas,-25729.36
Ohio,-16971.38
Pennsylvania,-15559.96
Illinois,-12607.89
North Carolina,-7490.91
Colorado,-6527.86
Tennessee,-5341.69
Arizona,-3427.92
Florida,-3399.30
Oregon,-1190.47


In [58]:
%%sql

--Q4-Product Profitability Classification:
select d.product_name, round(sum(sales),0) as Total_sales , round(sum(profit),2),
            round(sum(profit) / NULLIF(sum(sales),0 )* 100,2) as margin_pct,
CASE 
    WHEN sum(profit) / NULLIF(sum(sales),0 ) >= 0.20 THEN 'High Margin'
    WHEN sum(profit) / NULLIF(sum(sales),0 ) >= 0.05 THEN 'Medium Margin'
    WHEN sum(profit) / NULLIF(sum(sales),0 ) >= 0     THEN 'Low Margin'
    ELSE  
        'Loss'
                END AS margin_category
FROM superstore.fact_sales f   
JOIN superstore.dim_product d on f.product_key = d.product_key
GROUP BY d.product_name
ORDER BY margin_pct DESC;


product_name,total_sales,round,margin_pct,margin_category
Tops Green Bar Computer Printout Paper,343,171.29,50.00,High Margin
Southworth Structures Collection,73,36.40,50.00,High Margin
Xerox 1890,245,122.35,50.00,High Margin
"Adams Telephone Message Book w/Frequently-Called Numbers Space, 400 Messages per Book",223,111.72,50.00,High Margin
Canon imageCLASS MF7460 Monochrome Digital Laser Multifunction Copier,3992,1995.99,50.00,High Margin
Avery 475,266,133.20,50.00,High Margin
Xerox 1984,13,6.35,49.00,High Margin
Acco Glide Clips,20,9.60,49.00,High Margin
Xerox 1983,30,14.65,49.00,High Margin
Personal Creations Ink Jet Cards and Labels,321,157.51,49.00,High Margin


In [59]:
%%sql

--Q5-Year-over-Year Profit Growth by Category:
WITH yearly_profit AS(SELECT d.year, p.category,round(sum(profit),2) as total_profit
FROM superstore.fact_sales f
JOIN superstore.dim_date d on f.order_date_key = d.date_key
JOIN superstore.dim_product p on f.product_key = p.product_key
GROUP BY d.year , p.category
)
SELECT cur.category, cur.YEAR, round(cur.total_profit/NULLIF(prev.total_profit,0)*100,2) as growth_pct
FROM yearly_profit cur
LEFT JOIN yearly_profit prev on prev.category = cur.category AND prev.YEAR= cur.YEAR-1
ORDER BY cur.category, cur.YEAR DESC;


category,year,growth_pct
Furniture,2019,43.37
Furniture,2018,230.83
Furniture,2017,55.25
Furniture,2016,None
Office Supplies,2019,113.33
Office Supplies,2018,139.69
Office Supplies,2017,111.09
Office Supplies,2016,None
Technology,2019,127.43
Technology,2018,118.71


In [60]:
%%sql

--Q6-Top 10 Customers by Total Sales
select c.customer_name, c.segment, 
                                count(sales_key)as total_Orders,
                                round(SUM(profit),2)as total_profit,
                                round(sum(sales),0) as Total_sales 
FROM superstore.fact_sales f
JOIN superstore.dim_customer c on c.customer_key = f.customer_key 
GROUP BY c.customer_key , c.segment
ORDER BY Total_sales DESC
LIMIT 10;

customer_name,segment,total_orders,total_profit,total_sales
Sean Miller,Home Office,15,-1980.74,25043
Tamara Chand,Corporate,12,8981.32,19052
Raymond Buch,Consumer,18,6976.10,15117
Tom Ashbrook,Home Office,10,4703.79,14596
Adrian Barton,Consumer,20,5444.81,14474
Ken Lonsdale,Consumer,29,806.86,14175
Sanjit Chand,Consumer,22,5757.41,14142
Hunter Lopez,Consumer,11,5622.43,12873
Sanjit Engle,Consumer,19,2650.68,12209
Christopher Conant,Consumer,11,2177.05,12129


In [61]:
%%sql

--Q7-Segments | Segment Performance Comparison
SELECT c.segment,
                COUNT(DISTINCT c.customer_key) as total_customers,
                                count(sales_key)as total_Orders,
                                round(SUM(profit),2)as total_profit,
                                round(sum(sales),0) as Total_sales, 
                                round(AVG(sales),2) as avg_order_value
FROM superstore.fact_sales as f
JOIN superstore.dim_customer as c on c.customer_key = f.customer_key
GROUP BY c.segment
ORDER BY Total_sales DESC;

segment,total_customers,total_orders,total_profit,total_sales,avg_order_value
Consumer,409,5191,134119.21,1161401,223.73
Corporate,236,3020,91979.13,706146,233.82
Home Office,148,1783,60298.68,429653,240.97


In [62]:
%%sql

--Q8-Customers with Above - Average Return Rate:-
WITH customer_return_rate AS (
    SELECT c.customer_key, customer_name ,  count(sales_key) as total_orders,
    SUM(CASE 
        WHEN is_returned THEN 1 ELSE 0 END)returned_orders,
    round(SUM(CASE 
        WHEN is_returned THEN 1 ELSE 0 END)::NUMERIC / count(sales_key) * 100 , 2) as  return_rate_pct
FROM superstore.fact_sales as f
JOIN superstore.dim_customer c on c.customer_key = f.customer_key
GROUP BY customer_name , c.customer_key 
)
SELECT * 
FROM customer_return_rate
WHERE return_rate_pct > (SELECT AVG(return_rate_pct)FROM customer_return_rate
)
ORDER BY return_rate_pct DESC;

customer_key,customer_name,total_orders,returned_orders,return_rate_pct
475,Hilary Holden,6,6,100.00
357,Roland Murray,2,2,100.00
46,Sandra Glassco,11,9,81.82
390,Andy Yotov,10,8,80.00
253,Joni Blumstein,8,6,75.00
504,Ted Butterfield,18,13,72.22
271,Greg Hansen,3,2,66.67
779,Deanra Eno,9,6,66.67
33,Cari Sayre,8,5,62.50
363,Brian Derr,5,3,60.00


In [63]:
%%sql

--Q9-Repeat vs One-Time Customer Classification:-
SELECT c.customer_key , c.segment, count(distinct order_id) as distinct_orders , sum(sales) as total_sales,
            case 
                WHEN count(distinct order_id) = 1 THEN 'One-Time Customer'
                WHEN count(distinct order_id) BETWEEN 2 AND 5 THEN 'Repeat Customer'
                ELSE 'Loyal Customer'
END AS customer_type
from superstore.fact_sales f
JOIN superstore.dim_customer c on c.customer_key = f.customer_key
GROUP BY c.customer_key , c.segment
ORDER BY distinct_orders DESC;

customer_key,segment,distinct_orders,total_sales,customer_type
464,Consumer,17,5478.0608,Loyal Customer
278,Consumer,13,6760.815,Loyal Customer
536,Corporate,13,2964.823,Loyal Customer
301,Consumer,13,8025.707,Loyal Customer
712,Corporate,13,3406.8395,Loyal Customer
4,Consumer,13,3154.855,Loyal Customer
292,Consumer,13,3086.907,Loyal Customer
592,Corporate,13,2846.705,Loyal Customer
91,Consumer,12,3913.418,Loyal Customer
463,Corporate,12,3247.642,Loyal Customer


In [64]:
%%sql

--Q10-Customers Who Never Returned a Product:
SELECT 
DISTINCT customer_id,customer_name, segment
from superstore.dim_customer c
join superstore.fact_sales f on c.customer_key = f.customer_key
where not exists
            (select * from superstore.fact_sales f
            where f.customer_key=c.customer_key and f.is_returned = true)
order by customer_name;

customer_id,customer_name,segment
AB-10015,Aaron Bergman,Consumer
AH-10030,Aaron Hawkins,Corporate
AS-10045,Aaron Smayling,Corporate
AB-10060,Adam Bellavance,Home Office
AH-10075,Adam Hart,Corporate
AS-10090,Adam Shillingsburg,Consumer
AS-10135,Adrian Shami,Home Office
AB-10150,Aimee Bixby,Consumer
AB-10165,Alan Barnes,Consumer
AD-10180,Alan Dominguez,Home Office


In [65]:
%%sql

--Q11-Monthly Sales Trend for a Specific Year:
SELECT 
month, 
month_name,
count(sales_key) as total_orders,
round(SUM(sales),0) as monthly_sales,
round(SUM(profit),2) as monthly_profit
FROM superstore.fact_sales f
JOIN superstore.dim_date d on d.date_key = f.order_date_key
WHERE year = 2019
GROUP BY month, month_name
ORDER BY month;


month,month_name,total_orders,monthly_sales,monthly_profit
1,january,155,43971,7140.44
2,february,107,20301,1613.87
3,march,238,58872,14751.89
4,april,203,36522,933.29
5,may,242,44261,6342.58
6,june,245,52982,8223.34
7,july,226,45264,6952.62
8,august,218,63121,9040.96
9,september,459,87867,10991.56
10,october,298,77777,9275.28


In [66]:
%%sql

--Q12-Quarterly Sales Comparison:
WITH quarterly_sales AS (
    SELECT 
        quarter, 
        year,
        count(sales_key) as total_orders,
        round(SUM(sales),0) as quarterly_sales,
        round(SUM(profit),2) as quarterly_profit
    FROM superstore.fact_sales f
    JOIN superstore.dim_date d on d.date_key = f.order_date_key
    GROUP BY quarter, year
)
select quarter,year,quarterly_sales,
round(quarterly_sales / sum(quarterly_sales) OVER(PARTITION BY year) * 100, 2) as pct_of_year_sales
from quarterly_sales
ORDER BY year, quarter;


quarter,year,quarterly_sales,pct_of_year_sales
1,2016,74448,15.37
2,2016,86539,17.87
3,2016,143633,29.66
4,2016,179628,37.09
1,2017,68852,14.63
2,2017,89124,18.94
3,2017,130260,27.68
4,2017,182297,38.74
1,2018,93237,15.30
2,2018,136082,22.34


In [67]:
%%sql

--Q13-Shipping Mode Performance:
SELECT 
    sm.ship_mode,
    count(sales_key) as total_orders,
    round(SUM(sales),0) as total_sales,
    round(SUM(profit),2) as total_profit,
    round(AVG(sales),2) as avg_order_value,
    round(AVG(sd.full_date - od.full_date),2) as avg_shipping_time,
    case
        when AVG(sd.full_date - od.full_date) <= 1 THEN 'Fast'
        when AVG(sd.full_date - od.full_date) <= 4 THEN 'Medium'
        else 'Slow'
    end as shipping_speed_category
FROM superstore.fact_sales f
JOIN superstore.dim_ship_mode sm on sm.ship_mode_key = f.ship_mode_key
join superstore.dim_date as od on od.date_key = f.order_date_key
join superstore.dim_date as sd on sd.date_key = f.ship_date_key
GROUP BY sm.ship_mode
order by avg_shipping_time DESC;

ship_mode,total_orders,total_sales,total_profit,avg_order_value,avg_shipping_time,shipping_speed_category
Standard Class,5968,1358216,164088.79,227.58,5.01,Slow
Second Class,1945,459194,57446.64,236.09,3.24,Medium
First Class,1538,351428,48969.84,228.50,2.18,Medium
Same Day,543,128363,15891.76,236.40,0.04,Fast


In [47]:
%%sql

--Q14-Products with Declining Year-over-Year Sales:
WITH yearly_product_sales AS (
SELECT 
        p.product_key,
        p.product_name,
        d.year,
        round(sum(f.sales),0) as total_sales
FROM superstore.fact_sales f
join superstore.dim_product p on p.product_key = f.product_key
join superstore.dim_date d on d.date_key = f.order_date_key
group by p.product_key, p.product_name, d.year
)

SELECT cur.product_name, cur.year, prev.total_sales as previous_year_sales, cur.total_sales as current_year_sales,
    round((cur.total_sales - prev.total_sales) / prev.total_sales * 100, 2) AS change_pct
from yearly_product_sales cur
join yearly_product_sales prev ON prev.product_key = cur.product_key 
and prev.year = cur.year - 1
where cur.total_sales < prev.total_sales
ORDER BY change_pct ASC;


product_name,year,previous_year_sales,current_year_sales,change_pct
Zebra GX420t Direct Thermal/Thermal Transfer Printer,2018,5522,265,-95.20
Newell 334,2018,317,16,-94.95
Manco Dry-Lighter Erasable Highlighter,2018,58,3,-94.83
Avery Durable Slant Ring Binders With Label Holder,2019,58,3,-94.83
"Dixon Prang Watercolor Pencils, 10-Color Set with Brush",2019,77,4,-94.81
GBC Ibimaster 500 Manual ProClick Binding System,2019,12861,761,-94.08
"Premier Elliptical Ring Binder, Black",2017,295,18,-93.90
Newell 351,2017,46,3,-93.48
9-3/4 Diameter Round Wall Clock,2017,168,11,-93.45
KI Adjustable-Height Table,2017,731,52,-92.89


In [46]:
%%sql

--Q15-Top 5 States by Total Profitability:
SELECT 
    l.state,l.region,
    count(distinct f.sales_key) as total_orders,
    round(SUM(f.sales),0) as total_sales,
    round(SUM(f.profit),2) as total_profit,
    round(SUM(f.profit) / NULLIF(SUM(f.sales), 0) * 100, 2) AS profit_margin_pct
FROM superstore.fact_sales f
JOIN superstore.dim_location l on l.location_key = f.location_key
GROUP BY l.state , l.region
ORDER BY total_profit DESC
LIMIT 5;

state,region,total_orders,total_sales,total_profit,profit_margin_pct
California,West,2001,457688,76381.39,16.69
New York,East,1128,310876,74038.55,23.82
Washington,West,506,138641,33402.65,24.09
Michigan,Central,255,76270,24463.19,32.07
Virginia,South,224,70637,18597.95,26.33


In [68]:
%%sql

--Optimize queries for performance:-
-- Foreign Keys
CREATE INDEX idx_fact_customer   
ON superstore.fact_sales(customer_key);
CREATE INDEX idx_fact_product    
ON superstore.fact_sales(product_key);
CREATE INDEX idx_fact_location   
ON superstore.fact_sales(location_key);
CREATE INDEX idx_fact_orderdate  
ON superstore.fact_sales(order_date_key);
CREATE INDEX idx_fact_shipdate   
ON superstore.fact_sales(ship_date_key);

-- is_returned, order_id
CREATE INDEX idx_fact_returned  
 ON superstore.fact_sales(is_returned);
CREATE INDEX idx_fact_orderid   
 ON superstore.fact_sales(order_id);

-- dim_date 
CREATE INDEX idx_date_year       
ON superstore.dim_date(year);

RuntimeError: (psycopg2.errors.DuplicateTable) relation "idx_fact_customer" already exists

[SQL: CREATE INDEX idx_fact_customer
ON superstore.fact_sales(customer_key);]
(Background on this error at: https://sqlalche.me/e/21/f405)


In [48]:
%%sql
--Test the performance of a query using EXPLAIN ANALYZE:
EXPLAIN ANALYZE
SELECT * FROM superstore.fact_sales WHERE customer_key = 5;

QUERY PLAN
Index Scan using idx_fact_customer on fact_sales (cost=0.29..12.48 rows=11 width=69) (actual time=0.159..0.170 rows=6.00 loops=1)
Index Cond: (customer_key = 5)
Index Searches: 1
Buffers: shared hit=7
Planning Time: 0.208 ms
Execution Time: 0.295 ms
